### Web Scraping RTVE

In [6]:
# En este caso se utiliza scraping porque el RSS de RTVE no estaba actualizado.
# Se recogen noticias de tres secciones: Nacional, Internacional y Cultura.
# Para cada noticia se extraen el enlace, la fecha, el título, el subtítulo y el contenido.
# Los resultados se guardan por día y también en un archivo acumulado sin duplicados.

import requests
from bs4 import BeautifulSoup
import json
import re
from urllib.parse import urljoin
from pathlib import Path
from datetime import date

HEADERS = {"User-Agent": "Mozilla/5.0"}

CATEGORIAS = [
    ("https://www.rtve.es/noticias/espana/", "Nacional"),
    ("https://www.rtve.es/noticias/internacional/", "Internacional"),
    ("https://www.rtve.es/noticias/cultura/", "Cultura")
]

BASE_DIR = Path.home() / "Desktop" / "RTVE_news"

In [ ]:
# Limpia espacios y saltos de línea
def clean_text(text):
    if not text:
        return None
    return re.sub(r"\s+", " ", text).strip()


# Limpia el título
def clean_title(title):
    if not title:
        return None
    title = clean_text(title)
    return re.sub(r"^\d{1,2}:\d{2}\s*min", "", title).strip()


# Descarga una página
def get_page_soup(url):
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


# Extrae enlaces de una sección
def extract_latest_links(category_url, categoria, max_items=15):
    soup = get_page_soup(category_url)
    articles = soup.find_all("article")

    results = []
    seen_links = set()

    for art in articles:
        a_tag = art.find("a", href=True)
        if not a_tag:
            continue

        link = urljoin(category_url, a_tag["href"])

        if "/noticias/" not in link or link in seen_links:
            continue

        title_tag = art.find(["h2", "h3"])
        title = title_tag.get_text(" ", strip=True) if title_tag else a_tag.get_text(" ", strip=True)
        title = clean_title(title)

        if not title:
            continue

        seen_links.add(link)
        results.append({
            "Link": link,
            "Periódico": "RTVE",
            "Título": title,
            "Categoría": categoria
        })

        if len(results) == max_items:
            break

    return results


# Obtiene la fecha
def extract_fecha(soup):
    time_tag = soup.find("time")

    if time_tag:
        raw = time_tag.get("datetime") or time_tag.get_text(" ", strip=True)
        match = re.search(r"\d{4}-\d{2}-\d{2}", raw)
        if match:
            return match.group(0)

    meta = soup.find("meta", attrs={"property": "article:published_time"})
    if meta and meta.get("content"):
        match = re.search(r"\d{4}-\d{2}-\d{2}", meta["content"])
        if match:
            return match.group(0)

    return ""


# Obtiene el subtítulo si aparece
def extract_subtitle(soup):
    meta_desc = soup.find("meta", attrs={"name": "description"})

    if meta_desc and meta_desc.get("content"):
        return clean_text(meta_desc["content"])

    return None


# Extrae texto de la noticia
def extract_contenido(soup, subtitulo=None):
    article = soup.find("article")
    paragraphs = article.find_all("p") if article else soup.find_all("p")

    texts = []

    for p in paragraphs:
        txt = clean_text(p.get_text(" ", strip=True))
        if txt and len(txt) > 40:
            texts.append(txt)

    if subtitulo and texts and texts[0] == subtitulo:
        texts = texts[1:]

    return " ".join(texts[:2]) if texts else ""


# Completa los datos de una noticia
def enrich_news_item(item):
    soup = get_page_soup(item["Link"])

    fecha = extract_fecha(soup)
    subtitulo = extract_subtitle(soup)
    contenido = extract_contenido(soup, subtitulo)

    return {
        "Link": item["Link"],
        "Periódico": item["Periódico"],
        "Fecha": fecha,
        "Título": item["Título"],
        "Subtítulo": subtitulo,
        "Categoría": item["Categoría"],
        "Contenido": contenido
    }


# Carga JSON si existe
def load_json_if_exists(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


# Guarda JSON
def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# Elimina duplicados por enlace
def deduplicate_by_link(items):
    unique = {}
    for item in items:
        unique[item["Link"]] = item
    return list(unique.values())

In [ ]:
# Proceso principal
def main():
    today_str = date.today().isoformat()

    BASE_DIR.mkdir(parents=True, exist_ok=True)

    today_dir = BASE_DIR / today_str
    today_dir.mkdir(parents=True, exist_ok=True)

    all_news_today = []

    for category_url, categoria in CATEGORIAS:
        print(f"Recogiendo {categoria}...")

        latest_links = extract_latest_links(category_url, categoria, max_items=15)

        category_news = []

        for item in latest_links:
            try:
                full_item = enrich_news_item(item)

                if full_item["Fecha"] != today_str:
                    continue

                category_news.append(full_item)
                all_news_today.append(full_item)

                if len(category_news) == 5:
                    break

            except Exception as e:
                print(f"Error en {item['Link']}: {e}")

        category_filename = categoria.lower() + ".json"
        save_json(today_dir / category_filename, category_news)

    daily_file = today_dir / f"rtve_{today_str}.json"
    save_json(daily_file, all_news_today)

    master_file = BASE_DIR / "RTVE.json"
    old_news = load_json_if_exists(master_file)
    merged_news = deduplicate_by_link(old_news + all_news_today)
    save_json(master_file, merged_news)

    print("\nGuardado completo.")
    print(f"Carpeta del día: {today_dir}")
    print(f"Archivo diario: {daily_file}")
    print(f"Archivo maestro: {master_file}")
    print(f"Noticias de hoy: {len(all_news_today)}")
    print(f"Noticias acumuladas (sin duplicados): {len(merged_news)}")


if __name__ == "__main__":
    main()

Recogiendo Nacional...
Recogiendo Internacional...
Recogiendo Cultura...

Guardado completo.
Carpeta del día: /Users/an/Desktop/RTVE_news/2026-04-27
Archivo diario: /Users/an/Desktop/RTVE_news/2026-04-27/rtve_2026-04-27.json
Archivo maestro: /Users/an/Desktop/RTVE_news/RTVE.json
Noticias de hoy: 12
Noticias acumuladas (sin duplicados): 82
